> **Run-Notebook `qwen3-4b-base-ablation`** (erzeugt von `scripts/make_ablation_notebook.py`, nicht von Hand bearbeiten): Modell
> `Qwen/Qwen3-4B-Base`, Drive-Ordner `MyDrive/ValueHeadsPythia/qwen3-4b` (derselbe wie im Overlap-Notebook). Laufzeit:
> **eine A100** (Colab Pro), *Laufzeit → Alle ausführen*.

# Kausaler Test: Ablation der Wert-Head × ICL-Überlappung (`Qwen/Qwen3-4B-Base`)

**Frage.** Die Top-15-%-HH-„helpful“-Probe-Heads überlappen den benannten 33-Head-ICL-Schaltkreis L20-27
(arXiv 2611.05344; Overlap-Notebook: 21 von 33 im Base-, 23 von 33 im Instruct-Modell, angereichert nach Holm). Ist das
kausal? **Stützt** sich das Modell für seine eigene Helpful-Präferenz auf diese Heads (*Reliance*), oder ist „helpful“
dort nur **dekodierbar** (*Expression*; Goodfire, „Shaping & Monitoring Models through their Internal Representations“)?

**Rolle Base:** Hier läuft die Positivkontrolle (Gate). Das Instruct-Notebook [`qwen3-4b-ablation`](https://colab.research.google.com/github/Erikiss/ValueHeads/blob/main/notebooks/runs/qwen3-4b-ablation.ipynb) ist parallel ausführbar; es wartet in Stufe 0 auf das Gate dieses Notebooks (`results/ablation/qwen3-4b-base/gate.json`, höchstens `GATE_WAIT_MIN` Minuten).

**Ablauf** (`python -m valueheads_pythia.cli ablate`, Spezifikation `docs/ABLATION_DESIGN.md`):
1 Laufzeit · 2 Installation/Projekt/Drive · 3 Konfiguration · 4 **Stufe 0 – Gate**: Eingaben (Probe-Tabelle aus dem
Overlap-Lauf, Census, Batterie), HH-Split (Prompt-Cluster-disjunkt zu Probe-Train/-Val; means_ref 256, means_probe 256,
EVAL 1000), Hook-Selbsttest, Referenzmittel, eingefrorene Kopf-Sätze und Zufallsziehungen (vor der ersten EVAL-Rechnung),
M2-Probes, Boden-Check, Positivkontrolle (CR20-Knockout auf der ICL-Batterie gegen 20 Zufallsmengen) · 5 Pilot + ETA ·
6 **Stufe 1 – primär**: mean-Ablation von C33 und O an den Antwortpositionen, je 100 layer- und KV-gematchte
Zufallsmengen, Holm über {C33, O} · 7 Stufe 2 – sekundär (M3 Umkehrtest, weitere Sätze, M4 WikiText-Schaden,
mean_last, M5 Selbstreparatur, M7 Kovariaten, resample, dir_mean, gauss)
· 8 Zusammenfassung.

**Budget** (Schätzung aus Scouting-Tokenzahlen, Rate nicht gemessen; der Pilot in Zelle 5 druckt die echte ETA):

| | Bedingungen | Tokens | A100-h bei 5k / 8k / 10k tok/s | CU (×6) bei 8k |
|---|---|---|---|---|
| Stufe 0 Base (inkl. Gate, 21 ICL-Bedingungen) | ~26 | ~2.1 M | 0.12 / 0.07 / 0.06 (+ Modell-Laden ~0.2) | ~2 |
| Stufe 1 (je Modell) | 203 | ~44.5 M | 2.5 / 1.5 / 1.2 | ~9 |
| Stufe 2 Base | ~640 | ~83 M | 4.6 / 2.9 / 2.3 | ~17 |
| Stufe 2 Instruct (inkl. feste Identität 105, CR20 21) | ~770 | ~100 M | 5.6 / 3.5 / 2.8 | ~21 |
| **Summe Base / Instruct** | | ~129 M / ~145 M | **7.2 / 4.5 / 3.6** bzw. **8.0 / 5.0 / 4.0** | **~27 / ~30** |

Reichen die CU nicht: Stufe 0 + Stufe 1 (≈ 2 h, ≈ 12 CU) beantworten die Primärfrage; Stufe 2 folgt der
Prioritätsreihenfolge (Arme in Zelle 3 abschaltbar). **Jede Zelle ist idempotent:** jede Bedingung ist eine eigene Datei
auf Drive; nach einem Verbindungsabbruch einfach *Alle ausführen*, fertige Bedingungen werden übersprungen.

**Datenschutz:** HH-Inhalte werden nie ausgegeben (nur IDs, Hashes, Zahlen); vor dem Speichern trotzdem *Bearbeiten →
Alle Ausgaben löschen*.

## 1. Laufzeit-Check

GPU-Typ → `TIER` (`cpu` / `t4` / `l4` / `a100`), RAM/Disk, Versionen, `HF_HOME` auf der lokalen Disk
(`/content/hf_cache`, nicht auf Drive: FUSE ist für große Safetensors unzuverlässig). `SMOKE=True` schaltet auf die
kleine Konfiguration (winzige Teilmengen, nur CPU-taugliche Zellen); auf einer CPU-Laufzeit ist das automatisch der
Fall.

In [ ]:
# 1. Laufzeit-Check: GPU-Typ -> TIER, RAM / Disk, Versionen, HF_HOME
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# SMOKE: tiny subsets, CPU-compatible cells only (env VH_NB_SMOKE=1, or set it by hand; CPU runtimes always use it)
SMOKE = os.environ.get("VH_NB_SMOKE", "0") == "1"

import torch

HAS_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else None
DEVICE = "cuda" if HAS_CUDA else "cpu"


def tier_of(gpu_name):
    """Colab tier from the GPU name: cpu / t4 / l4 / a100 (an unknown GPU is treated like a T4)."""
    if not gpu_name:
        return "cpu"
    name = gpu_name.upper()
    if "A100" in name or "H100" in name:
        return "a100"
    if "L4" in name:
        return "l4"
    return "t4"


TIER = tier_of(GPU_NAME)
if TIER == "cpu" and "VH_NB_SMOKE" not in os.environ:
    SMOKE = True

HF_HOME = os.environ.get("HF_HOME") or ("/content/hf_cache" if IN_COLAB else os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = HF_HOME
os.makedirs(HF_HOME, exist_ok=True)


def version_of(package):
    """Installed version of a distribution without importing it (transformers is pinned in cell 2)."""
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return "-"


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk = shutil.disk_usage("/content" if IN_COLAB else os.getcwd())
print(f"Colab: {IN_COLAB} | GPU: {GPU_NAME or 'none'} -> TIER = {TIER} | DEVICE = {DEVICE} | SMOKE = {SMOKE}")
print(f"RAM {ram_gb:.1f} GB | Disk frei {disk.free / 1e9:.0f} GB | CPUs {os.cpu_count()} | {platform.platform()}")
print(f"python {platform.python_version()} | torch {torch.__version__} | transformers {version_of('transformers')} | "
      f"datasets {version_of('datasets')} | scikit-learn {version_of('scikit-learn')}")
if HAS_CUDA:
    print(f"VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | bf16 unterstuetzt: {torch.cuda.is_bf16_supported()}")
if IN_COLAB and not os.environ.get("HF_TOKEN"):
    try:                                              # optional Colab secret HF_TOKEN (read token): higher Hub rate limits for parallel sessions
        from google.colab import userdata
        _tok = userdata.get("HF_TOKEN")
        if _tok:
            os.environ["HF_TOKEN"] = _tok
    except Exception:
        pass
print(f"HF_HOME = {HF_HOME} | HF_TOKEN gesetzt: {bool(os.environ.get('HF_TOKEN'))}")

## 2. Installation, Projekt, Drive

Drive mounten, Projekt holen (Default `git clone` von GitHub), gepinnte Abhängigkeiten installieren. `VH_ROOT` zeigt auf
`MyDrive/ValueHeadsPythia/qwen3-4b`: dort liegen die Probe-Tabellen des Overlap-Laufs (`results/heads/`) und die
Ergebnisse dieses Tests (`results/ablation/`). `PROJECT_SOURCE = "drive"` erwartet das Projekt unter
`MyDrive/ValueHeadsPythia/valueheads_pythia` (wie im Master-Notebook).

In [ ]:
# 2. Installation (gepinnt), Projekt (Drive-Kopie | git clone), Drive mounten, Ordnerstruktur (DESIGN 6.4)
DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT = f"{DRIVE_MOUNT}/MyDrive/ValueHeadsPythia/qwen3-4b"          # DESIGN 6.4 layout root on Drive
PROJECT_SOURCE = "git"        # "git": clone GITHUB_URL (default) | "drive": project folder on Drive
GITHUB_URL = "https://github.com/Erikiss/ValueHeads.git"  # used with PROJECT_SOURCE == "git"
DRIVE_PROJECT = f"{DRIVE_MOUNT}/MyDrive/ValueHeadsPythia/valueheads_pythia"                # README.md, valueheads_pythia/, scripts/, ... on Drive
LOCAL_PROJECT = "/content/valueheads_pythia"                     # code runs from the local disk


def find_project(start):
    """Project root (the directory holding valueheads_pythia/cli.py) at `start` or one of its parents, else None."""
    d = os.path.abspath(start)
    for _ in range(4):
        if os.path.exists(os.path.join(d, "valueheads_pythia", "cli.py")):
            return d
        d = os.path.dirname(d)
    return None


def pip_install(*spec):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec], check=True)


if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if find_project(LOCAL_PROJECT) and PROJECT_SOURCE == "git" and os.path.isdir(os.path.join(LOCAL_PROJECT, ".git")):
        subprocess.run(["git", "-C", LOCAL_PROJECT, "pull", "--ff-only"], check=False)   # pick up fixes without a runtime reset
    if not find_project(LOCAL_PROJECT):
        if PROJECT_SOURCE == "git":
            subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, LOCAL_PROJECT], check=True)
        else:
            # the project root is the directory holding valueheads_pythia/cli.py: MyDrive/ValueHeadsPythia/valueheads_pythia/
            # (documented layout) or MyDrive/ValueHeadsPythia/ itself when the project CONTENTS were copied there
            src = find_project(DRIVE_PROJECT) or find_project(DRIVE_ROOT)
            if not src:
                raise FileNotFoundError(f"Projekt nicht gefunden: Ordner nach {DRIVE_PROJECT} kopieren (Drive) oder "
                                        f"PROJECT_SOURCE = 'git' und GITHUB_URL setzen")
            shutil.copytree(src, LOCAL_PROJECT,
                            ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", "vh_runs", "features", "results", "figures",
                                                          "hf_cache", "*.npy", "*.npz"))
    PROJECT = LOCAL_PROJECT
    if not os.path.exists(os.path.join(PROJECT, "requirements.txt")):
        raise FileNotFoundError(f"{PROJECT}/requirements.txt fehlt: {PROJECT} ist kein vollstaendiger Projektordner (README.md, "
                                "requirements.txt, valueheads_pythia/, scripts/, data/ erwartet)")
    pip_install("-r", os.path.join(PROJECT, "requirements.txt"))      # torch>=2.1, transformers>=4.56,<6, datasets>=2.19, ...
    if TIER in ("t4", "l4"):
        pip_install("bitsandbytes")                                    # optional: 4-bit judges (ArmoRM, TruthfulQA judges)
    VH_ROOT = DRIVE_ROOT                                               # results/ figures/ on Drive
    VH_FEATURES = "/content/features"                                  # features on the local disk (volatile)
    DRIVE_FEATURE_CACHE = f"{DRIVE_ROOT}/features"                     # capped feature cache on Drive
else:
    PROJECT = os.environ.get("VH_NB_PROJECT") or find_project(os.getcwd()) or find_project(os.path.join(os.getcwd(), ".."))
    if not PROJECT:
        raise FileNotFoundError("Projekt nicht gefunden: VH_NB_PROJECT setzen oder das Notebook aus dem Projekt starten")
    VH_ROOT = os.path.abspath(os.environ.get("VH_ROOT") or os.path.join(PROJECT, "vh_runs"))
    VH_FEATURES = os.path.abspath(os.environ.get("VH_FEATURES") or os.path.join(VH_ROOT, "features"))
    DRIVE_FEATURE_CACHE = os.path.join(VH_ROOT, "features_cache")      # stands in for the Drive cache outside Colab

os.environ.update(VH_ROOT=VH_ROOT, VH_FEATURES=VH_FEATURES)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from valueheads_pythia import config

P = config.paths(VH_ROOT)
for d in [P["heads"], P["directions"], P["eval"], P["judge"], P["sweep"], P["heatmaps"], P["figures_sweep"],
          P["figures_causal"], VH_FEATURES, DRIVE_FEATURE_CACHE]:
    os.makedirs(d, exist_ok=True)
print(f"PROJECT      {PROJECT}\nVH_ROOT      {VH_ROOT}  (results/, figures/)\nVH_FEATURES  {VH_FEATURES}\n"
      f"Feature-Cache {DRIVE_FEATURE_CACHE} (Cap 4 GB)\nHF_HOME      {HF_HOME}")
from valueheads_pythia.cli import COMMANDS

print("CLI-Befehle:", ", ".join(COMMANDS))

## 3. Konfiguration

Budget-Knöpfe (Defaults = Design): `N_EVAL` EVAL-Paare, `R_C33` / `R_O` Zufallsmengen der Primärfamilie, `R_SMALL` für
CminusO / P_band, `R_SEC` für die sekundären Arme, `BOOTSTRAP` Cluster-Bootstrap-Ziehungen, `ARMS` = Arme der Stufe 2
(an/aus). Kleinere R sind Präfixe der eingefrorenen Ziehungen (nichts Gerechnetes geht verloren); größere R brauchen
`--extend_random` (hängt Ziehungen an, die vorhandenen bleiben). **`N_EVAL` ist kein Präfix:** es legt Items,
Partner-Antworten (hh3) und damit den Schlüssel jeder HH-Bedingung fest. Vor Stufe 0 wählen; eine spätere Änderung
rechnet jede HH-Bedingung neu und wird mit Exit 2 abgelehnt, solange `CHANGE_N_EVAL = False` ist (nach dem Pilot kostet
sie nur `clean__hh3` und die zwei Pilot-Bedingungen). `BATCH_TOKENS` / `MAX_BATCH` gehen ebenfalls in die Schlüssel der
Bedingungen ein: zwischen den Stufen nicht ändern. Überschreibbar per Umgebung (CPU-Smoke): `VH_NB_MODELS` (erster Eintrag ersetzt
`MODEL`), `VH_NB_ARMS` (Komma-Liste), `VH_NB_SMOKE`. Die Zelle prüft, ob die Probe-Tabelle des Overlap-Laufs auf Drive
liegt, und lädt einmal den Census (sha256 gegen den gepinnten Commit).

In [ ]:
# 3. Konfiguration: Modell, Budget-Knoepfe, Arme, Gate-Wartezeit, CLI-Argumente
from valueheads_pythia import icl_sets

MODEL = "Qwen/Qwen3-4B-Base"
if os.environ.get("VH_NB_MODELS"):                         # e.g. CPU smoke: Qwen/Qwen3-0.6B-Base
    MODEL = [m.strip() for m in os.environ["VH_NB_MODELS"].split(",") if m.strip()][0]
SHORT = config.model_short_name(MODEL)
N_EVAL = 1000                      # EVAL pairs; enters every HH condition key: choose before stage 0
CHANGE_N_EVAL = False              # True once to accept a changed N_EVAL (every HH condition is recomputed)
R_C33 = 100                        # random sets of the primary family (C33)
R_O = 100                          # random sets of the primary family (O)
R_SMALL = 50                       # CminusO, P_band (own null)
R_SEC = 20                         # secondary arms and the gate
BOOTSTRAP = 10000                  # cluster-bootstrap resamples (M2: min(B, 1000))
ARMS = {"m3": True, "mean_sets": True, "m4": True, "mean_last": True, "m5": True, "m7": True, "resample": True,
        "dir_mean": True, "gauss": True, "fixed_identity": False}
if os.environ.get("VH_NB_ARMS"):
    _on = {a.strip() for a in os.environ["VH_NB_ARMS"].split(",") if a.strip()}
    ARMS = {a: a in _on for a in ARMS}
BATCH_TOKENS = 8192 if HAS_CUDA else 2048     # enters the condition keys: keep it fixed between the stages
MAX_BATCH = 64 if HAS_CUDA else 16
GATE_WAIT_MIN = 90                 # instruct: minutes to wait for the Base gate in stage 0
PROGRESS = [] if (IN_COLAB and not SMOKE) else ["--no_progress"]
NOTE = "SMOKE auf winzigen Teilmengen (CPU): nur Funktionspruefung, kein Befund" if SMOKE else None

COMMON = ["--model", MODEL, "--device", DEVICE, "--batch_tokens", BATCH_TOKENS, "--max_batch", MAX_BATCH,
          "--gate_wait_min", GATE_WAIT_MIN, *PROGRESS]
if SMOKE:
    COMMON += ["--smoke", "--note", NOTE]        # F28 sizes (8 EVAL, 8 + 8 MEANS, R = 2, B = 200), advisory gate
else:
    COMMON += ["--n_eval", N_EVAL, "--r_c33", R_C33, "--r_o", R_O, "--r_small", R_SMALL, "--r_sec", R_SEC,
               "--bootstrap", BOOTSTRAP]
if CHANGE_N_EVAL:
    COMMON += ["--change_n_eval"]
ABL = os.path.join(P["results"], "ablation")
RUN_DIR = os.path.join(ABL, SHORT)
ICL_ID = icl_sets.resolve_icl_model(SHORT)[0]
GATE_JSON = os.path.join(ABL, ICL_ID, "gate.json")

info = config.MODELS.get(SHORT, {})
print(f"MODEL {MODEL} -> {SHORT}: L/H/KV/D = {info.get('layers')}/{info.get('heads')}/{info.get('kv_heads')}/"
      f"{info.get('head_dim')}, numerics {config.dtype_policy(MODEL, DEVICE)} on {DEVICE}; ICL-Identitaet {ICL_ID}"
      + (" (Rolle Base: Gate hier)" if ICL_ID == SHORT else " (Rolle Instruct: wartet auf das Base-Gate)"))
table = config.heads_path(P, SHORT, None, "helpful", "pythia")
print(f"Probe-Tabelle helpful/pythia: {table} -> {'vorhanden' if os.path.exists(table) else 'FEHLT'}"
      + ("" if os.path.exists(table) else (" (SMOKE: synthetische Tabelle)" if SMOKE else
                                           " -> zuerst das Overlap-Notebook (qwen3-4b-overlap) laufen lassen")))
census = icl_sets.census_sets(ICL_ID)         # env VH_ICL_REPO, cache, or the pinned GitHub URL; sha256-checked
print(f"ICL census {ICL_ID}: {census.origin} (pinned {census.pinned}), " + ", ".join(f"{k} {len(v)}" for k, v in census.roles.items()))
print(f"N_EVAL {N_EVAL}, R = {R_C33}/{R_O}/{R_SMALL}/{R_SEC}, B = {BOOTSTRAP}, batch {BATCH_TOKENS}/{MAX_BATCH}"
      + (" | SMOKE: Groessen von --smoke (F28)" if SMOKE else ""))
print("Arme Stufe 2:", ", ".join(a for a, on in ARMS.items() if on))


def read_json(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def fmt(x, d=3, sign=True):
    return "–" if x is None else (f"{x:+.{d}f}" if sign else f"{x:.{d}f}")

QUIET_PATTERNS = ("Warning: You are sending", "Loading weights", "UserWarning", "warnings.warn", "FutureWarning")


def run(*args, check=True, extra_env=None):
    """Runs `python -m valueheads_pythia.cli <args>` in a subprocess (fresh process per step: the model's memory is
    released afterwards), streams its output into the notebook and raises on a non-zero exit code."""
    cmd = [sys.executable, "-m", "valueheads_pythia.cli", *map(str, args)]
    env = dict(os.environ, HF_HOME=HF_HOME, VH_ROOT=VH_ROOT, VH_FEATURES=VH_FEATURES, PYTHONUNBUFFERED="1",
               OMP_NUM_THREADS=str(os.cpu_count() or 1), TOKENIZERS_PARALLELISM="false", **(extra_env or {}))
    t0 = time.time()
    print("$ " + " ".join(cmd[2:]), flush=True)
    proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            if not any(p in line for p in QUIET_PATTERNS):
                print(line, end="", flush=True)
        rc = proc.wait()
    except BaseException:
        # cell interrupt: stop the step instead of leaving it running behind the notebook (a second run of the same
        # cell would otherwise write into the same feature / result directories at the same time)
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"[abgebrochen nach {time.time() - t0:.0f}s: {' '.join(cmd[2:4])}]", flush=True)
        raise
    print(f"[exit {rc}, {time.time() - t0:.0f}s]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {' '.join(cmd[2:])}")
    return rc


run("--help")

## 4. Stufe 0 – Gate

Prüft die Eingaben (Probe-Tabelle: Modus pythia, AUC, `n_train = 20 000`; Census; ICL-Batterie-Datei), schreibt bzw.
verifiziert den HH-Split, lädt das Modell (bf16 + sdpa), Hook-Selbsttest (Referenz = eigene saubere Werte reproduziert die
Logits; leerer Satz = kein Hook; Lokalität; Antwortpositionen [b − 1, Ende] unabhängig aus Länge und Grenze nachgerechnet,
Prompt und Padding unberührt, erstes Antwort-Token reagiert; Kausalität), Referenzmittel auf means_ref, M2-Trainingsmerkmale auf
means_probe, **friert die Kopf-Sätze und Zufallsziehungen ein** (`results/ablation/{SHORT}_frozen_sets.json`, vor der
ersten EVAL-Rechnung), trainiert die M2-Probes, rechnet `clean__hh3` und den Boden-Check und – im Base-Modell – die
Positivkontrolle: Kalibrierung (Census-Ziehung, Seed 1, gegen D188_summary), CR20-Knockout an der Query-Position der
frischen Batterie (Seed 2) gegen 20 layer- und KV-gematchte Zufallsmengen. **STOPP**, wenn das Gate nicht besteht
(Exit 3): dann wird kein Helpfulness-Befund berichtet, der STOPP-Bericht (`results/ablation/{SHORT}_ablation.md`) liegt
trotzdem auf Drive, und die Zellen danach brechen ab. Ein volles Instruct-Notebook akzeptiert nur ein bestandenes Gate
eines vollen Base-Laufs, kein beratendes Smoke-Gate.

In [ ]:
# 4. Stufe 0: Selbsttest, Split, Referenzen, eingefrorene Saetze, M2-Probes, Boden-Check, Gate (idempotent)
rc0 = run("ablate", *COMMON, "--stage", 0, check=False)
sel = read_json(os.path.join(RUN_DIR, "selftest.json")) or {}
for name in ("C33", "WholeBand"):
    st = sel.get(name)
    if st:
        print(f"Hook-Selbsttest {name}: {'ok' if st['ok'] else 'FEHLER'} (" + ", ".join(
            f"{k} {v['value']:.1e}" for k, v in st["checks"].items()) + ")")
fl = read_json(os.path.join(RUN_DIR, "floor.json"))
if fl:
    print("Boden-Check:", fl["text_de"])
gate = read_json(GATE_JSON)
if gate:
    rnd = gate.get("random") or {}
    print(f"Gate ({'PASS' if gate['passed'] else ('beratend (Smoke)' if gate.get('advisory') else 'STOPP')}): CR20 "
          f"Δdret {fmt(gate['delta'])} [{fmt(gate['lo'])}, {fmt(gate['hi'])}] Nats; Zufall Mittel "
          f"{fmt(rnd.get('rand_mean'))}, Min {fmt(rnd.get('rand_min'))} (R = {rnd.get('R')})")
    print("  ", gate.get("reason_de"))
if rc0 == 3 and os.path.exists(os.path.join(ABL, f"{SHORT}_ablation.md")):
    print("STOPP-Bericht:", os.path.join(ABL, f"{SHORT}_ablation.md"))
if rc0 != 0:
    raise RuntimeError(f"Stufe 0 beendet mit Exit {rc0} (2 Eingaben fehlen oder N_EVAL geaendert, 3 Gate nicht "
                       "bestanden / fehlt / nur Smoke, 4 Selbsttest, 5 eingefrorene Saetze weichen ab) – siehe Ausgabe oben")

## 5. Pilot + ETA

Zwei Bedingungen (`mean__hh3__C33` und ihre erste Zufallsmenge) messen die Rate (Tokens/s); daraus die ETA und die
CU-Schätzung (~6 CU je A100-Stunde) für alle ausstehenden Bedingungen der Stufen 1 und 2. Zu langsam? `R_*` oder `ARMS`
in Zelle 3 verkleinern (kleinere R sind Präfixe der eingefrorenen Ziehungen; nichts Gerechnetes geht verloren). Eine
kleinere `N_EVAL` ist **kein** Präfix: sie rechnet jede HH-Bedingung neu (jetzt nur `clean__hh3` und die zwei
Pilot-Bedingungen) und braucht dafür einmal `CHANGE_N_EVAL = True`; sonst bricht der Lauf mit Exit 2 ab.

In [ ]:
# 5. Pilot: zwei Bedingungen, gemessene Rate -> ETA / CU je Stufe und Arm
run("ablate", *COMMON, "--stage", 1, "--pilot", "--arms", *[a for a, on in ARMS.items() if on])
pilot = read_json(os.path.join(RUN_DIR, "pilot.json"))
if pilot:
    print(f"\nGemessen ({pilot['device']}, {pilot.get('gpu') or 'CPU'}): " + ", ".join(
        f"{k} {v:.0f} tok/s" for k, v in sorted(pilot["rates"].items())))
    tot = pilot["total"]
    print(f"Ausstehend: {tot['n']} Bedingungen, {tot['tokens'] / 1e6:.1f} M Tokens, ~{tot['hours']:.2f} h, "
          f"~{tot['cu_a100']:.0f} CU (A100, {pilot['cu_per_a100_hour']} CU/h)")

## 6. Stufe 1 – primär

mean-Ablation (feste label-neutrale Referenz je Head, an allen Antwortpositionen) von **C33** und **O = C33 ∩ Top-k**
in Variante hh3 (gewählt, abgelehnt, fremde Antwort), je 100 layer- und KV-gematchte Zufallsmengen. Primäre Familie
je Modell: Δ margin_mean (gepaart je Item) für C33 und O, einseitiges empirisches p in Richtung „senkt“, Holm (α = 0.05).

In [ ]:
# 6. Stufe 1: primaere Zellen (C33, O + je R Zufallsmengen), dann Statistik + Bericht
run("ablate", *COMMON, "--stage", 1)
rep = read_json(os.path.join(ABL, f"{SHORT}_ablation.json"))
if rep:
    print("\nPrimaere Familie (Holm, alpha = 0.05):")
    for r in rep["primary"]:
        print(f"  {r['set']:4s} ({r['n_heads']} Heads): Δ margin_mean {fmt(r['observed'])} [{fmt(r['lo'])}, {fmt(r['hi'])}]"
              f" | Zufall {fmt(r['rand_mean'])} ± {fmt(r['rand_std'], sign=False)} (R = {r['R']}) | p einseitig "
              f"{r['p_one'] if r['p_one'] is None else round(r['p_one'], 4)} | Holm-p "
              f"{r['p_holm'] if r['p_holm'] is None else round(r['p_holm'], 4)} | {r.get('verdict', '')}")
    for line in rep["verdict"]:
        print("  " + line)

## 7. Stufe 2 – sekundär

Arme in Prioritätsreihenfolge: **m3** (ICL-Umkehrtest: C33, O, P_band, TopK_all, WholeBand, Census-Sätze),
**mean_sets** (CminusO, P_band, P_DoM, C33-Leser/-Bremsen, TopK_all, WholeBand), **m4** (WikiText-2-Schaden, 50 000
Tokens), **mean_last** (Referenz am letzten Token; auf M1 strukturell 0, nur M2), **m5** (Selbstreparatur: direkter Effekt
per DLA gegen Gesamteffekt), **m7** (Prompt-Wiederverwendung, Längen-Terzile, Prompt-Tausch), **resample** (Spender-Paar),
**dir_mean** (nur die DoM-Richtung je Head, gegen zufällige Richtungen), **gauss** (normgleiches Rauschen) und im
Instruct-Lauf **fixed_identity** (Base-Sätze auf dem Instruct-Modell). Alles sekundär (p ohne Befund) oder deskriptiv.

In [ ]:
# 7. Stufe 2: sekundaere Arme (idempotent; abgeschaltete Arme in ARMS bleiben aus)
arms = [a for a, on in ARMS.items() if on]
if arms:
    run("ablate", *COMMON, "--stage", 2, "--arms", *arms)
else:
    print("keine Arme aktiviert")

## 8. Zusammenfassung

Bericht aus den gespeicherten Bedingungen (ohne Modell): `results/ablation/{SHORT}_ablation.{json,md}`,
`…_conditions.csv`, `…_tests.csv`. Angezeigt wird der Kopf bis zu den sekundären Tabellen (vollständig in der Datei).
**Vor dem Speichern: Bearbeiten → Alle Ausgaben löschen.**

In [ ]:
# 8. Zusammenfassung: Bericht neu schreiben und anzeigen (Kopf bis "Sekundaer")
from IPython.display import Markdown, display

run("ablate", *COMMON, "--summary")
md_path = os.path.join(ABL, f"{SHORT}_ablation.md")
with open(md_path) as f:
    text = f.read()
cut = text.find("\n## Sekundär")
display(Markdown(text[:cut] if cut > 0 else text))
print("Ergebnisse:", ABL)
for suffix in ("_ablation.json", "_ablation.md", "_ablation_conditions.csv", "_ablation_tests.csv", "_frozen_sets.json"):
    path = os.path.join(ABL, SHORT + suffix)
    print(f"  {path} ({os.path.getsize(path) / 1e3:.0f} kB)" if os.path.exists(path) else f"  {path} (fehlt)")
n_cond = len([f for f in os.listdir(os.path.join(RUN_DIR, "conditions")) if f.endswith(".json")]) \
    if os.path.isdir(os.path.join(RUN_DIR, "conditions")) else 0
print(f"  {n_cond} Bedingungsdateien unter {os.path.join(RUN_DIR, 'conditions')}")
print("\n*** Vor dem Speichern des Notebooks: Bearbeiten -> Alle Ausgaben loeschen (Edit -> Clear all outputs). ***")

---
**Erinnerung:** *Bearbeiten → Alle Ausgaben löschen* (Edit → Clear all outputs), bevor das Notebook gespeichert, auf
Drive abgelegt oder geteilt wird.